# Relativistic charged particle

*em · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/em/relativistic_particle)

The Lorentz force written covariantly, $dU^\mu/d\tau = (q/m)\,F^\mu{}_\nu U^\nu$: the field tensor's second index is lowered with the Minkowski metric and contracted with the four-velocity in one einsum. Integrating in proper time shows fast particles circling more slowly than Newton predicts, and a constant electric field pushing a particle ever closer to — but never past — the speed of light.

**Vector form**

$$\frac{d(\gamma m\mathbf v)}{dt} = q\,(\mathbf E + \mathbf v\times\mathbf B),\qquad \frac{d(\gamma m)}{dt} = q\,\mathbf E\cdot\mathbf v$$

**Index (tensor) form**

$$\frac{dU^\mu}{d\tau} = \frac{q}{m}\,F^{\mu}{}_{\nu}\,U^\nu,\qquad F^{\mu}{}_{\nu} = F^{\mu\alpha}\,\eta_{\alpha\nu},\qquad U^\mu U_\mu = 1$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $X^\mu,\ U^\mu$ | `y[:, 0], y[:, 1]` | (P, 4) | four-position $(t, x, y, z)$ and four-velocity $\gamma(1, \mathbf v)$ of particle $p$ |
| $\eta_{\mu\nu}$ | `metric()` | (4, 4) | Minkowski metric $\operatorname{diag}(+1, -1, -1, -1)$, $c = 1$ |
| $F^{\mu\nu}$ | `field_tensor(E, B, eps)` | (4, 4) | $F^{0i} = -E_i$, $F^{ij} = -\varepsilon_{ijk}B_k$ |
| $F^{\mu}{}_{\nu}$ | `einsum('ma,an->mn', F, eta)` | (4, 4) | repeated $\alpha$ ⇒ summed: second index lowered |
| $\tfrac{q}{m}F^{\mu}{}_{\nu}U^\nu$ | `einsum('p,mn,pn->pm', qm, F_mixed, U)` | (P, 4) | repeated $\nu$ ⇒ summed: four-force on every particle |
| $U^\mu U_\mu$ | `einsum('...m,mn,...n->...', U, eta, U)` | (T, P) | Minkowski length of $U$, must stay 1 |

Units with $c = 1$. The equations are integrated in each particle's proper time $\tau$ with RK4 and then resampled onto the lab time $t = X^0$ for the animation. Cyclotron: the magnetic force does no work, so $\gamma$ stays constant, but the orbit is $\gamma$ times larger and $\gamma$ times slower than the Newtonian circle ($\omega = qB/(\gamma m)$). Accelerate: from rest in a uniform $E$ the motion is hyperbolic, $x = (\sqrt{1 + (at)^2} - 1)/a$ with $a = qE/m$, so $v \to 1$ while Newton's $\tfrac12 at^2$ would overtake light after $t = 2/a$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/em/relativistic_particle/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import numpy as np

from problems._common.runtime import cli

from . import plot, solve, tensors
from .equation import field_tensor, lower_second, minkowski_norm


def run(params: dict, out_dir: Path) -> Path:
    config, strength, t_max = params["config"], params["strength"], params["t_max"]
    eta = tensors.metric()  # (4, 4)   tiny tensors: CPU float64
    E, B = tensors.fields(config, strength)  # (3,), (3,)
    F_mixed = lower_second(field_tensor(E, B, tensors.levi_civita()), eta)  # F^μ_ν (4, 4)
    qm, y0 = tensors.particles(config, params["n_particles"], params["v_max"])  # (P,), (P, 2, 4)

    tau, traj = solve.solve(y0, qm, F_mixed, t_max)  # (T,), (T, P, 2, 4)
    t = np.linspace(0.0, t_max, 300)
    lab = solve.to_lab_time(traj, t)
    newton = solve.newtonian(config, qm, y0, strength, t)

    U = traj[:, :, 1]  # (T, P, 4)
    drift = ((minkowski_norm(U, eta) - 1) / U[..., 0] ** 2).abs().max()  # relative to γ², the size of each term
    print(f"{len(tau)} proper-time steps, largest relative drift of U^mu U_mu from 1: {drift:.1e}")
    if config == "cyclotron":
        v = (y0[:, 1, 2] / y0[:, 1, 0]).tolist()
        labels = [f"v = {s:.2f}, γ = {g:.2f}" for s, g in zip(v, y0[:, 1, 0].tolist())]
    else:
        labels = [f"qE/m = {a * strength:g}" for a in qm.tolist()]
    for label, g in zip(labels, lab["gamma"][-1]):
        print(f"{label:28s} gamma at t_max = {g:.3f}")
    return plot.animate(lab, newton, config, labels, strength, out_dir / "relativistic_particle.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/em/relativistic_particle/equation.py
r"""Physics → tensor translation.

The covariant Lorentz force (c = 1, metric η = diag(+1, −1, −1, −1)) for the four-velocity U^μ = γ(1, v):

    \frac{dU^\mu}{d\tau} = \frac{q}{m} F^{\mu}{}_{\nu} U^\nu,      \frac{dX^\mu}{d\tau} = U^\mu

τ is the particle's own (proper) time. The field tensor has two upper indices,

    F^{0i} = -E_i,    F^{i0} = E_i,    F^{ij} = -\varepsilon_{ijk} B_k ,

and to contract with U^ν its second index is lowered with the metric:

    F^{\mu}{}_{\nu} = F^{\mu\alpha} \eta_{\alpha\nu}      (α summed)

For a batch of particles p the force is one einsum('p,mn,pn->pm'): ν is repeated, so it is summed.
Written out, the space part is γ(E + v × B) and the time part E·v, the rate of work done on the particle.

The length of the four-velocity is fixed:  U^\mu U_\mu = \eta_{\mu\nu} U^\mu U^\nu = γ²(1 − v²) = 1.
"""

import torch


def field_tensor(E: torch.Tensor, B: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    r"""F^{μν} of shape (4, 4) from E_i, B_k (3,) and the Levi-Civita symbol ε_{ijk} (3, 3, 3)."""
    F = torch.zeros(4, 4, dtype=E.dtype)
    F[0, 1:] = -E  # F^{0i} = -E_i
    F[1:, 0] = E  # F^{i0} = +E_i
    F[1:, 1:] = -torch.einsum("ijk,k->ij", eps, B)  # F^{ij} = -ε_ijk B_k
    return F


def lower_second(F: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    r"""F^μ_ν = F^{μα} η_{αν}  (α summed).   (4, 4) → (4, 4)"""
    return torch.einsum("ma,an->mn", F, eta)


def four_force(U: torch.Tensor, qm: torch.Tensor, F_mixed: torch.Tensor) -> torch.Tensor:
    r"""dU^μ/dτ = (q/m)_p F^μ_ν U^ν_p  (ν summed).   U: (P, 4), qm: (P,) → (P, 4)"""
    return torch.einsum("p,mn,pn->pm", qm, F_mixed, U)


def rhs(y: torch.Tensor, qm: torch.Tensor, F_mixed: torch.Tensor) -> torch.Tensor:
    """d/dτ of the state y (P, 2, 4): y[:, 0] = X^μ (position), y[:, 1] = U^μ (four-velocity)."""
    X_dot = y[:, 1]
    U_dot = four_force(y[:, 1], qm, F_mixed)
    return torch.stack([X_dot, U_dot], dim=1)


def minkowski_norm(U: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    r"""U^μ U_μ = η_{μν} U^μ U^ν  (both summed).   (..., 4) → (...)"""
    return torch.einsum("...m,mn,...n->...", U, eta, U)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/em/relativistic_particle/tensors.py
"""Build the concrete tensors: metric, Levi-Civita symbol, fields and the particle batch."""

import itertools

import torch

DTYPE = torch.float64  # U^0 = γ grows large under constant E: keep the extra digits
EVEN = [(0, 1, 2), (1, 2, 0), (2, 0, 1)]  # cyclic permutations of (x, y, z)


def metric() -> torch.Tensor:
    """η_{μν} = diag(+1, −1, −1, −1) of shape (4, 4)."""
    return torch.diag(torch.tensor([1.0, -1.0, -1.0, -1.0], dtype=DTYPE))


def levi_civita() -> torch.Tensor:
    """ε_ijk of shape (3, 3, 3): +1 for even permutations, −1 for odd ones, 0 when an index repeats."""
    eps = torch.zeros(3, 3, 3, dtype=DTYPE)
    for i, j, k in itertools.permutations(range(3)):
        eps[i, j, k] = 1.0 if (i, j, k) in EVEN else -1.0
    return eps


def fields(config: str, strength: float) -> tuple[torch.Tensor, torch.Tensor]:
    """E_i and B_i, shape (3,) each: B along z for "cyclotron", E along x for "accelerate"."""
    E, B = torch.zeros(3, dtype=DTYPE), torch.zeros(3, dtype=DTYPE)
    if config == "cyclotron":
        B[2] = strength
    else:
        E[0] = strength
    return E, B


def particles(config: str, n: int, v_max: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (q/m (P,), initial state y0 (P, 2, 4)) with every particle starting at the origin at t = 0.

    cyclotron:  q/m = 1, speeds v_p from 0.3 up to v_max along +y, so U^μ = γ (1, 0, v, 0)
    accelerate: at rest, U^μ = (1, 0, 0, 0), with q/m = 0.25, 0.5, 1, 2, ... (doubling)
    """
    y0 = torch.zeros(n, 2, 4, dtype=DTYPE)
    if config == "cyclotron":
        v = torch.linspace(0.3, v_max, n, dtype=DTYPE) if n > 1 else torch.tensor([v_max], dtype=DTYPE)
        gamma = 1 / torch.sqrt(1 - v**2)
        y0[:, 1, 0] = gamma  # U^0 = γ
        y0[:, 1, 2] = gamma * v  # U^y = γ v
        qm = torch.ones(n, dtype=DTYPE)
    else:
        y0[:, 1, 0] = 1.0  # at rest: U^μ = (1, 0, 0, 0)
        qm = 0.25 * 2.0 ** torch.arange(n, dtype=DTYPE)
    return qm, y0


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/em/relativistic_particle/solve.py
"""Integrate in proper time with RK4, then resample every particle onto a common lab-time grid."""

import numpy as np
import torch

from problems._common.integrators import rk4_step

from .equation import rhs


def solve(y0: torch.Tensor, qm: torch.Tensor, F_mixed: torch.Tensor, t_max: float, dtau: float = 0.01):
    """Return (τ (T,), trajectory (T, P, 2, 4)) — all P particles advance together in proper time τ.

    We step until every particle's lab clock X^0 has passed t_max. Fast particles' clocks run ahead
    (dt/dτ = γ), so the slowest particle decides when to stop.
    """
    ys = [y0]
    while ys[-1][:, 0, 0].min() < t_max:
        ys.append(rk4_step(lambda y, tau: rhs(y, qm, F_mixed), ys[-1], 0.0, dtau))
    return torch.arange(len(ys), dtype=y0.dtype) * dtau, torch.stack(ys)


def to_lab_time(traj: torch.Tensor, t: np.ndarray) -> dict[str, np.ndarray]:
    """Resample onto lab times t (F,): positions x (F, P, 3) and γ = U^0 (F, P).

    Each particle's clock runs at its own rate (dt/dτ = γ), so equal steps in τ are unequal steps in t.
    X^0 = t grows monotonically, so we can interpolate everything else as a function of it.
    """
    X = traj[:, :, 0].numpy()  # (T, P, 4)
    U = traj[:, :, 1].numpy()  # (T, P, 4)
    n_particles = X.shape[1]
    x = np.stack([[np.interp(t, X[:, p, 0], X[:, p, i]) for i in (1, 2, 3)] for p in range(n_particles)])
    gamma = np.stack([np.interp(t, X[:, p, 0], U[:, p, 0]) for p in range(n_particles)])
    # turns of the velocity vector in the x–y plane (it starts along +y, positive charges turn clockwise)
    angle = np.unwrap(np.arctan2(U[:, :, 2], U[:, :, 1]), axis=0)  # (T, P)
    turns = np.stack([np.interp(t, X[:, p, 0], (angle[0, p] - angle[:, p]) / (2 * np.pi)) for p in range(n_particles)])
    return {"t": t, "x": x.transpose(2, 0, 1), "gamma": gamma.T, "turns": turns.T}  # (F, P, 3), (F, P), (F, P)


def newtonian(config: str, qm: torch.Tensor, y0: torch.Tensor, strength: float, t: np.ndarray) -> np.ndarray:
    """What Newton would predict, positions (F, P, 3): circles at ω = qB/m, or x = ½ a t² with a = qE/m."""
    qm, v = qm.numpy(), (y0[:, 1, 1:] / y0[:, 1, :1]).numpy()  # q/m (P,), initial 3-velocity (P, 3)
    x = np.zeros((len(t), len(qm), 3))
    if config == "cyclotron":
        omega = qm * strength  # (P,)
        radius = v[:, 1] / omega  # start moving along +y, centre at (+r, 0)
        x[:, :, 0] = radius * (1 - np.cos(np.outer(t, omega)))
        x[:, :, 1] = radius * np.sin(np.outer(t, omega))
    else:
        x[:, :, 0] = 0.5 * np.outer(t**2, qm * strength)
    return x


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/em/relativistic_particle/plot.py
"""Animate relativistic (solid) against Newtonian (dashed) motion, with γ or the number of turns alongside."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif


def animate(lab: dict, newton: np.ndarray, config: str, labels: list[str], strength: float, path: Path) -> Path:
    t, x, gamma, turns = lab["t"], lab["x"], lab["gamma"], lab["turns"]  # (F,), (F, P, 3), (F, P), (F, P)
    n_particles = x.shape[1]
    colours = [f"C{p}" for p in range(n_particles)]
    fig, (ax, ax_right) = plt.subplots(1, 2, figsize=(7, 4))

    if config == "cyclotron":
        # left: orbits in the x–y plane; right: turns completed, Newton predicts ω = qB/m for everyone
        for p in range(n_particles):
            ax.plot(newton[:, p, 0], newton[:, p, 1], "--", color=colours[p], lw=0.8, alpha=0.6)
        ax.set_aspect("equal")
        lo, hi = min(x[..., :2].min(), newton[..., :2].min()), max(x[..., :2].max(), newton[..., :2].max())
        ax.set_xlim(lo - 0.3, hi + 0.3)
        ax.set_ylim(lo - 0.3, hi + 0.3)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_title("orbits in B_z (dashed: Newton)")
        horizontal, vertical, right = x[..., 0], x[..., 1], turns
        ax_right.plot(t, strength * t / (2 * np.pi), "k--", lw=1, label="Newton: ω = qB/m")
        ax_right.set_ylim(0, strength * t[-1] / (2 * np.pi) * 1.05)
        ax_right.set_ylabel("turns completed")
        ax_right.set_title("relativistic: ω = qB/(γm)")
    else:
        # left: x(t) can never cross the light line x = t; right: γ grows without limit
        ax.plot(t, t, "k:", lw=1, label="light: x = t")
        for p in range(n_particles):
            ax.plot(t, newton[:, p, 0], "--", color=colours[p], lw=0.8, alpha=0.6)
        ax.set_xlim(0, t[-1])
        ax.set_ylim(0, t[-1] * 1.05)
        ax.set_xlabel("t")
        ax.set_ylabel("x")
        ax.set_title("pushed by E_x (dashed: Newton ½at²)")
        horizontal, vertical, right = np.broadcast_to(t[:, None], gamma.shape), x[..., 0], gamma
        ax_right.set_ylim(0, gamma.max() * 1.05)
        ax_right.set_ylabel("γ = U⁰")
        ax_right.set_title("Lorentz factor γ(t)")

    trails = [ax.plot([], [], color=c, lw=1.6, label=lab)[0] for c, lab in zip(colours, labels)]
    dots = [ax.plot([], [], "o", ms=5, color=c)[0] for c in colours]
    curves = [ax_right.plot([], [], color=c, lw=1.6)[0] for c in colours]
    ax.legend(loc="upper left", fontsize=7)
    ax_right.set_xlim(0, t[-1])
    ax_right.set_xlabel("lab time t")
    if config == "cyclotron":
        ax_right.legend(loc="upper left", fontsize=8)

    def update(step: int):
        for p in range(n_particles):
            trails[p].set_data(horizontal[: step + 1, p], vertical[: step + 1, p])
            dots[p].set_data([horizontal[step, p]], [vertical[step, p]])
            curves[p].set_data(t[: step + 1], right[: step + 1, p])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'config': 'cyclotron',
    'strength': 1.0,
    'v_max': 0.95,
    'n_particles': 4,
    't_max': 25.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.em.relativistic_particle")]:
    del sys.modules[name]
main = importlib.import_module("problems.em.relativistic_particle.main")
problem_dir = Path("problems/em/relativistic_particle")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
